In [ ]:
import requests
import time
import ast
import re
import pickle
import logging
import pandas as pd
from typing import List, Dict, Tuple, Optional
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# RDKit imports
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit import RDLogger
import matplotlib.pyplot as plt

In [ ]:


# Suppress RDKit C++ warnings for unparseable/junk SMILES to keep the console clean
RDLogger.DisableLog('rdApp.*')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class LigandFilterTracker:
    """Tracks dropped chemical data to maintain full visibility into the curation."""
    def __init__(self):
        self.kegg_to_sid_failed = []
        self.sid_to_cid_failed = []
        self.cid_to_smiles_failed = []
        self.rdkit_parse_failed = []
        self.filtered_out_too_small = [] # MW < 100
        self.filtered_out_too_large = [] # MW > 900
        self.reactions_dropped = 0

    def print_summary(self):
        print("\n" + "="*50)
        print("🧪 LIGAND FILTERING DROP SUMMARY")
        print("="*50)
        print(f"Failed KEGG -> PubChem SID:     {len(self.kegg_to_sid_failed)}")
        print(f"Failed SID -> PubChem CID:      {len(self.sid_to_cid_failed)}")
        print(f"Failed CID -> SMILES:           {len(self.cid_to_smiles_failed)}")
        print(f"RDKit failed to parse SMILES:   {len(self.rdkit_parse_failed)}")
        print(f"Dropped (MW < 100 Da):          {len(self.filtered_out_too_small)}")
        print(f"Dropped (MW > 900 Da):          {len(self.filtered_out_too_large)}")
        print("-" * 50)
        print(f"Total Reactions Dropped from Manifest: {self.reactions_dropped}")
        print("="*50 + "\n")


class LigandProcessor:
    def __init__(self):
        self.session = requests.Session()
        # PubChem is strict: Max 5 requests per second, max 400 per minute
        retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
        self.session.mount('https://', HTTPAdapter(max_retries=retries))
        self.session.mount('http://', HTTPAdapter(max_retries=retries))
        self.tracker = LigandFilterTracker()

    def _make_request(self, url: str, sleep_time: float) -> Optional[str]:
        """Generic helper to handle both KEGG and PubChem with specific rate limits."""
        try:
            response = self.session.get(url, timeout=10)
            time.sleep(sleep_time) # Dynamically throttle based on the API
            
            if response.status_code == 200:
                return response.text.strip()
            return None
            
        except requests.exceptions.RequestException as e:
            logger.error(f"API request failed for {url}: {e}")
            return None

    def get_smiles_from_kegg(self, kegg_id: str) -> Optional[str]:
        """3-Step highly reliable conversion: KEGG ID -> SID -> CID -> SMILES"""
        
        # 1. KEGG -> PubChem SID (Bypassing the /link 400 error via the /get flat file)
        kegg_url = f"http://rest.kegg.jp/get/cpd:{kegg_id}"
        
        # KEGG allows faster requests, 0.1s is safe
        kegg_data = self._make_request(kegg_url, sleep_time=0.1) 
        
        if not kegg_data:
            self.tracker.kegg_to_sid_failed.append(kegg_id)
            return None
            
        # Use Regex to extract the PubChem SID from the DBLINKS block
        match = re.search(r'PubChem:\s*(\d+)', kegg_data)
        if not match:
            self.tracker.kegg_to_sid_failed.append(kegg_id)
            return None
            
        sid = match.group(1)

        # 2. PubChem SID -> PubChem CID
        cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/substance/sid/{sid}/cids/TXT"
        # PubChem is strict: 5 requests per second max. 0.22s sleep is required.
        cid = self._make_request(cid_url, sleep_time=0.22) 
        if not cid:
            self.tracker.sid_to_cid_failed.append(kegg_id)
            return None

        # 3. PubChem CID -> Canonical SMILES
        smiles_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/CanonicalSMILES/TXT"
        smiles = self._make_request(smiles_url, sleep_time=0.22)
        if not smiles:
            self.tracker.cid_to_smiles_failed.append(kegg_id)
            return None

        return smiles

    def is_drug_like(self, smiles: str, kegg_id: str) -> bool:
        """Applies RDKit filters for Molecular Weight."""
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            self.tracker.rdkit_parse_failed.append(kegg_id)
            return False
            
        mw = Descriptors.MolWt(mol)
        
        if mw < 50:
            self.tracker.filtered_out_too_small.append((kegg_id, mw))
            return False
        if mw > 900:
            self.tracker.filtered_out_too_large.append((kegg_id, mw))
            return False
            
        return True


def run_phase_2(df: pd.DataFrame, cache) -> Tuple[pd.DataFrame, Dict[str, str]]:
    """
    Takes the manifest from Phase 1, fetches/filters SMILES, and purges empty reactions.
    """
    logger.info("Starting Phase 2: Ligand Processing and Filtering...")
    processor = LigandProcessor()
    
    # 1. Extract all unique KEGG Compound IDs from the DataFrame
    # Note: Phase 1 stored 'Ligands' as a list of C-numbers in each row
    unique_kegg_ids = set()
    for ligands_list in df['Ligands']:
        unique_kegg_ids.update(ligands_list)
        
    logger.info(f"Found {len(unique_kegg_ids)} unique KEGG compounds to process.")
    
    # 2. Process each unique ligand
    valid_smiles_dict = {}
    for i, kegg_id in enumerate(unique_kegg_ids):
        if i % 50 == 0 and i > 0:
            logger.info(f"Processed {i}/{len(unique_kegg_ids)} compounds...")
        
        if kegg_id in cache:
            smiles = cache[kegg_id]
        else:
            smiles = processor.get_smiles_from_kegg(kegg_id)
            cache[kegg_id] = smiles
        if smiles and processor.is_drug_like(smiles, kegg_id):
            valid_smiles_dict[kegg_id] = smiles
        
        if i % 100 == 0:
            with open("smiles_dict_164.pkl", "wb") as f:
                pickle.dump(cache, f)

    # 3. Update the Manifest
    logger.info("Updating dataset manifest...")
    initial_row_count = len(df)
    
    # Replace the list of all KEGG IDs with a list of *only valid* SMILES strings
    df['Valid_SMILES'] = df['Ligands'].apply(
        lambda kegg_list: [valid_smiles_dict[k_id] for k_id in kegg_list if k_id in valid_smiles_dict]
    )
    
    # Drop rows where NO ligands passed the filter
    df_filtered = df[df['Valid_SMILES'].str.len() > 0].copy()
    
    processor.tracker.reactions_dropped = initial_row_count - len(df_filtered)
    processor.tracker.print_summary()
    
    logger.info(f"Phase 2 Complete. Dataset reduced from {initial_row_count} to {len(df_filtered)} viable reactions.")
    
    return df_filtered, valid_smiles_dict, processor.tracker

# Example Execution Hook:
# final_df_phase1 is the pandas DataFrame returned from our previous script
# clean_df, smiles_dictionary = run_phase_2(final_df_phase1)

In [ ]:
df = pd.read_csv("dataset_164.csv", index_col = 0)
if isinstance(df['Ligands'].iloc[0], str):
    df['Ligands'] = df['Ligands'].apply(ast.literal_eval)

In [ ]:
df.shape

In [ ]:
import pickle
smiles_dictionary = pickle.load(open("smiles_dict.pkl", "rb"))

In [ ]:
len(smiles_dictionary)

In [ ]:
clean_df, smiles_dictionary, tracker = run_phase_2(df, cache) 

In [ ]:
clean_df.to_csv("step_2_with_smiles_164.csv")

In [ ]:
clean_df.shape

In [ ]:
x = clean_df.explode("Valid_SMILES").drop_duplicates(["Valid_SMILES", "UniProt_ID"])[["Valid_SMILES", "UniProt_ID"]]

In [ ]:
x["Ligand"] = x["Valid_SMILES"].apply(lambda x: reverse_map[x])

In [ ]:
reverse_map = {v:k for k, v in smiles_dictionary.items()}

In [ ]:
y = x.groupby("Ligand")["UniProt_ID"].nunique().sort_values(ascending=False)

In [ ]:
(100 * y / y.sum()).head(10).round(3)

In [ ]:
(100 * y / y.sum()).head(10).sum()

In [ ]:
x.reset_index(drop=True).head(1)

In [ ]:
interaction = []
for i in range(1, 101, 1):
    interaction.append((100 * y / y.sum()).head(i).sum())

In [ ]:
plt.plot(interaction)
plt.xlabel("Number of ligands")
plt.ylabel("% of interaction covered")
plt.title(f"Total interactions covered: {round(interaction[-1])}")

In [ ]:
import pandas as pd

df = x
MAX_CAP = 100

print(f"Original Dataset Size: {len(df)} rows")
print(f"Original Unique Ligands: {df['Ligand'].nunique()}")
print(f"Original Unique Proteins: {df['UniProt_ID'].nunique()}\n")

# Step 1: Calculate global frequency of each ligand
ligand_counts = df['Ligand'].value_counts()
df['Ligand_Freq'] = df['Ligand'].map(ligand_counts)

# Step 2: Sort by Frequency (ASCENDING). Rare ligands are now at the top!
df_sorted = df.sort_values(by='Ligand_Freq', ascending=True)

# Step 3: THE BACKBONE. Get 1 row per protein.
# Because rare ligands are at the top, if a protein binds to both a rare drug AND ATP,
# drop_duplicates keeps the rare drug and ignores the ATP.
df_proteins = df_sorted.drop_duplicates(subset=['UniProt_ID'], keep='first')

# Step 4: THE ORPHANS. Did we miss any ligands?
covered_ligands = set(df_proteins['Ligand'])
missing_ligands = set(df['Ligand']) - covered_ligands

# Fetch 1 row for any missed ligands
df_missing = df[df['Ligand'].isin(missing_ligands)].drop_duplicates(subset=['Ligand'])

# Combine to form our essential foundation (~50,000 rows)
# 100% Proteins covered. 100% Ligands covered. Hubs heavily suppressed.
df_foundation = pd.concat([df_proteins, df_missing]).drop_duplicates()

# Step 5: THE FILL. Add more rows until we hit MAX_CAP for each ligand.
current_counts = df_foundation['Ligand'].value_counts()
df_unused = df.drop(df_foundation.index) # Don't sample rows we already have

extra_rows = []
for ligand, group in df_unused.groupby('Ligand'):
    current_n = current_counts.get(ligand, 0)
    needed = MAX_CAP - current_n
    
    if needed > 0:
        # Sample what we need, or take whatever is left if it's less than we need
        sampled = group.sample(n=min(needed, len(group)), random_state=42)
        extra_rows.append(sampled)

# Finalize the dataset
if extra_rows:
    df_extra = pd.concat(extra_rows)
    df_final = pd.concat([df_foundation, df_extra]).reset_index(drop=True)
else:
    df_final = df_foundation.reset_index(drop=True)

df_final = df_final.drop(columns=['Ligand_Freq']) # Clean up

print("--- After Level 1.5 Balancing ---")
print(f"New Dataset Size: {len(df_final)} rows")
print(f"Unique Ligands Preserved: {df_final['Ligand'].nunique()}")
print(f"Unique Proteins Preserved: {df_final['UniProt_ID'].nunique()}")

print("\nTop 5 Ligands now:")
print(df_final['Ligand'].value_counts().head(5))

In [ ]:
y = df_final.groupby("Ligand")["UniProt_ID"].nunique().sort_values(ascending=False)

In [ ]:
100 * (y / y.sum()).head(10)

In [ ]:
interaction = []
for i in range(1, 500, 1):
    interaction.append((100 * y / y.sum()).head(i).sum())

In [ ]:
plt.plot(interaction)
plt.xlabel("Number of ligands")
plt.ylabel("% of interaction covered")
plt.title(f"Total interactions covered: {round(interaction[-1])}")

In [ ]:
x["Ligand"].nunique(), x["UniProt_ID"].nunique()

In [ ]:
x.shape

In [ ]:
smiles_dictionary

In [ ]:
processor = LigandProcessor()

In [ ]:
url = f"http://rest.kegg.jp/get/cpd:C00008"
response = processor.session.get(url, timeout=10)

In [ ]:
kegg_data = response.text
match = re.search(r'PubChem:\s*(\d+)', kegg_data)

In [ ]:
sid = match.group(1)

# 2. PubChem SID -> PubChem CID
cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/substance/sid/{sid}/cids/TXT"
# PubChem is strict: 5 requests per second max. 0.22s sleep is required.
cid = processor._make_request(cid_url, sleep_time=0.22) 


In [ ]:
sid, cid

In [ ]:
print(response.text )

In [ ]:
import glob
import json

In [ ]:
len(files := glob.glob("alphafold_cif/*"))

In [ ]:
files[0]

In [ ]:
x = pd.read_csv("step_2_with_smiles.csv", index_col = 0)

In [ ]:
eval(x["Valid_SMILES"][0])

In [ ]:
x.iloc[0]["Ligands"]

In [ ]:
df.head(1)

In [ ]:
import requests

def get_protein_sequence(uniprot_id: str) -> str:
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    
    response = requests.get(url)
    
    if response.status_code != 200:
        raise ValueError(f"Failed to fetch data for UniProt ID: {uniprot_id}")
    
    fasta = response.text
    
    # Remove header and join sequence lines
    sequence = "".join(fasta.split("\n")[1:])
    
    return sequence




In [ ]:
import torch

In [ ]:
uniprot_to_seq = torch.load("/Users/aadi/Documents/Study material/protein_interaction_dataset_creation/uniprot_to_sequence.pt")

In [ ]:
uniprot_to_seq

In [ ]:
# Example
seq = get_protein_sequence("P69905")  # Hemoglobin subunit alpha
print(seq)

In [ ]:
df_final["sequence"] = df_final["UniProt_ID"].apply(lambda x: uniprot_to_seq.get(x))

In [ ]:
df_final.isna().sum()

In [ ]:
df_final[df_final["sequence"].isna()]["UniProt_ID"].nunique()

In [ ]:
import requests
from typing import List, Dict

def fetch_sequences_batch(uniprot_ids: List[str], batch_size: int = 500) -> Dict[str, str]:
    """
    Fetch sequences for a list of UniProt IDs in batches.
    
    Returns:
        dict: {uniprot_id: sequence}
    """
    base_url = "https://rest.uniprot.org/uniprotkb/stream"
    all_sequences = {}

    for i in range(0, len(uniprot_ids), batch_size):
        batch = uniprot_ids[i:i + batch_size]
        ids_str = " OR ".join(batch)  # UniProt query syntax
        
        params = {
            "format": "fasta",
            "query": f"accession:({ids_str})"
        }

        response = requests.get(base_url, params=params)
        response.raise_for_status()

        current_id = None
        for line in response.text.splitlines():
            if line.startswith(">"):
                # Example header: >sp|P69905|HBA_HUMAN ...
                current_id = line.split("|")[1]
                all_sequences[current_id] = ""
            else:
                all_sequences[current_id] += line.strip()

        print(f"Fetched batch {i // batch_size + 1}")

    return all_sequences


In [ ]:
new_sequence_ids = df_final[df_final["sequence"].isna()]["UniProt_ID"].unique()

In [ ]:
new_sequences = fetch_sequences_batch(new_sequence_ids, batch_size=24)

In [ ]:
len(new_sequences)

In [ ]:
for k, v in new_sequences.items():
    if len(v) <= 2:
        print(k)

In [ ]:
uniprot_to_seq.update(new_sequences)

In [ ]:
len(uniprot_to_seq)

In [ ]:
torch.save(uniprot_to_seq, 'uniprot_to_sequence_49700.pt')

In [ ]:
df_final["sequence"] = df_final["UniProt_ID"].apply(lambda x: uniprot_to_seq[x])

In [ ]:
df_final.to_parquet("dataset_164_step2.pq")